In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import yaml

BASE_PATH = "/content/drive/MyDrive/mosquito-robustness"

config_path = os.path.join(BASE_PATH, "configs/config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Config loaded.")

Config loaded.


In [3]:
from torchvision import datasets

In [4]:
#data augmentation
from torchvision import transforms

image_size = config["data"]["image_size"]

train_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(config["normalization"]["mean"],
                         config["normalization"]["std"])
])

test_transforms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(config["normalization"]["mean"],
                         config["normalization"]["std"])
])

In [5]:
!unzip -q "/content/drive/MyDrive/images.zip" -d "/content/"

base_dir = "/content/images"

train_dir = os.path.join(base_dir, "training")
test_dir  = os.path.join(base_dir, "testing")

print("Train classes:", os.listdir(train_dir))
print("Test classes:", os.listdir(test_dir))

Train classes: ['Culex Quinquefasciatus', 'Aedes Aegypti', 'Anopheles Arabiensis', 'Anopheles Freeborni', 'Anopheles Albimanus', 'Aedes Albopictus']
Test classes: ['Culex Quinquefasciatus', 'Aedes Aegypti', 'Anopheles Arabiensis', 'Anopheles Freeborni', 'Anopheles Albimanus', 'Aedes Albopictus']


In [6]:


full_train_dataset = datasets.ImageFolder(train_dir)
test_dataset_raw   = datasets.ImageFolder(test_dir)

print("Train samples:", len(full_train_dataset))
print("Test samples:", len(test_dataset_raw))
print("Classes:", full_train_dataset.classes)

Train samples: 2400
Test samples: 600
Classes: ['Aedes Aegypti', 'Aedes Albopictus', 'Anopheles Albimanus', 'Anopheles Arabiensis', 'Anopheles Freeborni', 'Culex Quinquefasciatus']


In [7]:
import torch
from torch.utils.data import random_split

val_size = int(0.1 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size

train_dataset_raw, val_dataset_raw = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train:", len(train_dataset_raw))
print("Val:", len(val_dataset_raw))
print("Test:", len(test_dataset_raw))

Train: 2160
Val: 240
Test: 600


In [8]:
train_dataset_raw.dataset.transform = train_transforms
val_dataset_raw.dataset.transform   = test_transforms
test_dataset_raw.transform          = test_transforms

train_dataset = train_dataset_raw
val_dataset   = val_dataset_raw
test_dataset  = test_dataset_raw

In [9]:
from torch.utils.data import DataLoader

batch_size = config["training"]["batch_size"]

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("DataLoaders ready.")

DataLoaders ready.


In [10]:
#sanity check
images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Labels shape:", labels.shape)

Batch shape: torch.Size([32, 3, 192, 192])
Labels shape: torch.Size([32])
